# GARDA (Grooming Alert and Detection Analysis)

**GARDA** adalah classifier teks yang mendeteksi indikasi *predatory grooming* online dalam percakapan chat berbahasa Inggris, dibangun di atas dataset PAN-2012. Pemrosesan data dan training model dilakukan di modul-modul terpisah di [GitHub repository](https://github.com/Zikabyte/garda-nlp) (`data_processing/`, `model_training/`). Notebook ini fokus ke bagian yang bisa dijalankan langsung tanpa akses ke dataset mentah: evaluasi model dan demo live.

**Scope (sengaja dibuat sederhana):** klasifikasi biner per pesan (suspicious vs normal), di-fine-tune pakai **SetFit** (`all-MiniLM-L6-v2`) alih-alih fine-tuning penuh model besar.

> **Catatan:** corpus PAN-2012 adalah data terbatas dan tidak boleh didistribusikan ulang (percakapan predator asli dari sting operation). Data ini **tidak** disertakan di repository atau notebook ini.

## Persiapan

Clone repo (kalau belum ada, misal saat dibuka pertama kali di Colab) dan install dependency.

In [ ]:
import os

if not os.path.exists("configs"):
    !git clone https://github.com/Zikabyte/garda-nlp.git
    %cd garda-nlp

%pip install -q -r requirements.txt

## 1. Evaluasi

Karena test set PAN-2012 asli tidak bisa disertakan dengan notebook ini, model dicek pakai `model_training/synthetic_conversations.py`. Model-nya sendiri di-load dari [Hugging Face Hub](https://huggingface.co/zikabyte/garda-setfit), bukan path lokal, jadi cell ini bisa jalan dari environment yang fresh.

In [ ]:
from demo.app import model
from model_training.evaluate import build_synthetic_df, evaluate

synthetic_df = build_synthetic_df()

probas = model.predict_proba(synthetic_df["context_text"].tolist(), as_numpy=True)[:, 1]
predictions = (probas >= 0.5).astype(int)

evaluate(synthetic_df["is_suspicious"], predictions, label="Synthetic conversations")

## 2. Demo Live

Menjalankan app Gradio aslinya (`demo/app.py`), dengan flow: paste percakapan, dapatkan risk score/band per baris dan ringkasan "baris paling berisiko". `share=True` membuat link publik sementara (berlaku ~72 jam).

In [ ]:
from demo.app import demo

demo.launch(share=True)